# ⚡ Infosys Springboard 7.0 · Batch 4 · Milestone 1
## User Authentication Module — Streamlit + JWT + ngrok + Gmail OTP

**Run order:** Cell 1 → 2 → 3 → 4. Before running, add these in **Colab Secrets (🔑 left sidebar)** and turn *Notebook access* ON for each:

| Secret | Value |
|---|---|
| `jwt_Key` | any long random string |
| `ngrok_authentication_HF` | token from the ngrok dashboard |
| `EMAIL_ADDRESS` | Gmail address that sends the OTP |
| `EMAIL_PASSWORD` | 16-character Gmail App Password |

### 1️⃣ Install dependencies

In [1]:
!pip install -q streamlit pyngrok bcrypt PyJWT plotly pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 62.5 MB/s eta 0:00:00


### 2️⃣ Check that all secrets are available (values are never printed)

In [2]:
from google.colab import userdata

# Each item: label -> secret names to look for (first one found is used)
SECRETS = {
    "JWT":   ["JWT_KEY", "JWT_SECRET"],
    "NGROK": ["ngrok_authentication_HF", "NGROK_AUTHTOKEN"],
    "EMAIL_ADDRESS":  ["EMAIL_ADDRESS"],
    "EMAIL_PASSWORD": ["EMAIL_PASSWORD"],
}

for label, names in SECRETS.items():
    found = None
    for n in names:
        try:
            if userdata.get(n):
                found = n
                break
        except Exception:
            pass
    print(f"{label:<15}", f"✅ found ({found})" if found else f"❌ MISSING - add one of: {names}")

JWT             ✅ found (JWT_KEY)
NGROK           ✅ found (ngrok_authentication_HF)
EMAIL_ADDRESS   ✅ found (EMAIL_ADDRESS)
EMAIL_PASSWORD  ✅ found (EMAIL_PASSWORD)


### 3️⃣ Write the Streamlit app to `app.py`

In [3]:
%%writefile app.py
"""Infosys Springboard 7.0 - Batch 4 - Milestone 1
User Authentication Module: Streamlit + JWT + SQLite + Gmail OTP
All secrets are read from environment variables (set from Colab Secrets by the launcher cell).
"""
import os, re, ssl, time, secrets, smtplib, sqlite3, hmac, datetime as dt
from contextlib import closing
from email.message import EmailMessage

import bcrypt, jwt
import pandas as pd
import plotly.graph_objects as pgo
import streamlit as st

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide",
                   initial_sidebar_state="auto")

# ───────────────────────── CONFIG (secrets come from env) ─────────────────────────
JWT_SECRET = os.environ.get("JWT_KEY", "")
EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD", "").replace(" ", "")

DB_PATH = "infosys_portal_m1.db"
JWT_HOURS = 2
OTP_TTL_SEC, OTP_MAX_TRIES, OTP_RESEND_SEC = 300, 5, 30
SECURITY_QUESTIONS = ["What is your pet's name?", "What is your mother's maiden name?",
                      "What is your favourite city?", "What was your first school's name?"]

if not JWT_SECRET:
    st.error("🔒 JWT_SECRET is missing. Add it in Colab Secrets (🔑) and re-run the launcher cell.")
    st.stop()

# ───────────────────────── STYLE ─────────────────────────
CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@500;600;700&family=Inter:wght@400;500;600;700&display=swap');
:root{--bg:#f9fcfc;--side:#e3f6f5;--card:#fff;--alt:#bae8e8;--ink:#272343;--text:#2d334a;--muted:#64748b;
      --accent:#ffd803;--accent-h:#e6c300;--ok:#34d399;--bad:#f87171;}
html,body,.stApp{background:var(--bg)!important;font-family:'Inter',sans-serif!important;color:var(--text)!important}
footer,div[data-testid="stDecoration"]{display:none!important}
header[data-testid="stHeader"]{background:transparent!important}
.block-container{padding:2rem 2.5rem 3rem!important;max-width:1180px}
h1,h2,h3,h4{font-family:'Poppins',sans-serif!important;color:var(--ink)!important}
label p{font-weight:600!important;color:var(--ink)!important}

div[data-baseweb="base-input"],div[data-baseweb="select"]>div{background:transparent!important;border:none!important}
div[data-baseweb="input"],div[data-baseweb="select"]{background:var(--card)!important;border:2px solid var(--ink)!important;border-radius:10px!important}
div[data-baseweb="input"]:focus-within{box-shadow:4px 4px 0 var(--ink)!important;border-color:var(--ink)!important}
input{color:var(--text)!important;-webkit-text-fill-color:var(--text)!important}

/* Forms become cards */
div[data-testid="stForm"]{background:var(--card);border:2px solid var(--ink)!important;border-radius:16px!important;
      padding:1.4rem 1.4rem .6rem!important;box-shadow:6px 6px 0 var(--ink)}

/* Buttons: default = white, primary = yellow */
div[data-testid="stButton"] button,div[data-testid="stFormSubmitButton"] button{
  width:100%;height:46px;border-radius:10px!important;border:2px solid var(--ink)!important;background:#fff!important;
  color:var(--ink)!important;font-weight:700!important;box-shadow:3px 3px 0 var(--ink);transition:all .15s ease}
div[data-testid="stButton"] button:hover,div[data-testid="stFormSubmitButton"] button:hover{
  transform:translate(-2px,-2px);box-shadow:5px 5px 0 var(--ink);background:var(--alt)!important}
div[data-testid="stButton"],div[data-testid="stFormSubmitButton"],div[data-testid="stElementContainer"]:has(>div[data-testid="stButton"]),div[data-testid="stElementContainer"]:has(div[data-testid="stFormSubmitButton"]){width:100%!important}
div[data-testid="stButton"] button[kind="primary"],div[data-testid="stFormSubmitButton"] button[kind="primaryFormSubmit"]{background:var(--accent)!important}
div[data-testid="stButton"] button[kind="primary"]:hover,div[data-testid="stFormSubmitButton"] button[kind="primaryFormSubmit"]:hover{background:var(--accent-h)!important}

section[data-testid="stSidebar"]{background:var(--side)!important;border-right:2px solid var(--ink)!important}
.card{background:var(--card);border:2px solid var(--ink);border-radius:14px;padding:22px;box-shadow:4px 4px 0 var(--alt)}
.hero{background:var(--ink);border-radius:18px;padding:34px 32px;color:#fff;box-shadow:6px 6px 0 var(--accent)}
.hero h1{color:var(--accent)!important;font-size:2.1rem!important;margin:.2rem 0 .4rem}
.hero p{color:var(--alt);font-size:.95rem;line-height:1.55}
.hero li{margin:.55rem 0;color:#fff;font-size:.93rem;list-style:none}
.hero ul{padding:0;margin:1.2rem 0 0}
.chip{display:inline-block;background:var(--accent);color:var(--ink);font-weight:700;border-radius:30px;padding:6px 16px;font-size:.85rem}
.title{font-family:'Poppins',sans-serif;font-size:1.5rem;font-weight:700;color:var(--ink);margin:0}
.sub{color:var(--muted);font-size:.9rem;margin:.1rem 0 1rem}
.banner{background:var(--ink);border-radius:16px;padding:22px 30px;display:flex;justify-content:space-between;align-items:center;margin-bottom:22px;flex-wrap:wrap;gap:10px}
.banner h1{color:var(--accent)!important;margin:0;font-size:1.5rem!important}
.banner .s{color:var(--alt);font-size:.85rem}
.kpi{text-align:center}.kpi .i{font-size:26px}.kpi .v{font-size:24px;font-weight:700;color:var(--ink)}.kpi .l{font-size:12px;font-weight:600;color:var(--muted)}
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)


def html(s: str):
    """Render HTML (strips indentation so Markdown never treats it as a code block)."""
    st.markdown("\n".join(l.strip() for l in s.splitlines()), unsafe_allow_html=True)


# ───────────────────────── DATABASE ─────────────────────────
def q(sql, args=(), one=False, write=False):
    with closing(sqlite3.connect(DB_PATH)) as c:
        cur = c.execute(sql, args)
        if write:
            c.commit()
            return cur.rowcount
        return cur.fetchone() if one else cur.fetchall()


q("""CREATE TABLE IF NOT EXISTS users(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL COLLATE NOCASE,
        email TEXT UNIQUE NOT NULL COLLATE NOCASE,
        password_hash TEXT NOT NULL,
        security_question TEXT NOT NULL,
        security_answer_hash TEXT NOT NULL,
        created_at TEXT NOT NULL)""", write=True)


def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bool(h) and bcrypt.checkpw(t.encode(), h.encode())
DUMMY_HASH = hash_txt("dummy-password")  # keeps login timing similar when the user doesn't exist


# ───────────────────────── VALIDATION (Step 8) ─────────────────────────
def valid_email(e: str) -> bool:
    """>=2 letters before @, >=2 letters between @ and final dot, >=2 letters after final dot."""
    if e.count("@") != 1:
        return False
    local, domain = e.split("@")
    if "." not in domain:
        return False
    mid, _, tld = domain.rpartition(".")
    letters = lambda s: sum(ch.isalpha() for ch in s)
    return (bool(re.fullmatch(r"[A-Za-z0-9._%+\-]+", local)) and bool(re.fullmatch(r"[A-Za-z0-9.\-]+", mid))
            and letters(local) >= 2 and letters(mid) >= 2 and tld.isalpha() and len(tld) >= 2)


def valid_username(u: str) -> bool:
    return bool(re.fullmatch(r"[A-Za-z0-9_.\-]{3,30}", u))


def password_problems(p: str):
    rules = [(len(p) >= 8, "at least 8 characters"), (re.search(r"[A-Z]", p), "an uppercase letter"),
             (re.search(r"[a-z]", p), "a lowercase letter"), (re.search(r"\d", p), "a number"),
             (re.search(r"[^A-Za-z0-9]", p), "a special symbol")]
    return [msg for ok, msg in rules if not ok]


def check_new_password(pw, cpw):
    """Returns an error string or None."""
    if not pw or not cpw:
        return "Both password fields are required."
    bad = password_problems(pw)
    if bad:
        return "Password needs " + ", ".join(bad) + "."
    if pw != cpw:
        return "Passwords do not match."
    return None


PW_HINT = "Min 8 chars · uppercase · lowercase · number · special symbol"


# ───────────────────────── JWT (Step 7) ─────────────────────────
def make_jwt(username, email, role="user"):
    now = dt.datetime.now(dt.timezone.utc)
    return jwt.encode({"sub": username, "email": email, "role": role, "iat": now,
                       "exp": now + dt.timedelta(hours=JWT_HOURS)}, JWT_SECRET, algorithm="HS256")


def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        return None


# ───────────────────────── EMAIL OTP ─────────────────────────
def send_otp_email(to_addr, otp):
    msg = EmailMessage()
    msg["Subject"] = "Your Infosys Portal verification code"
    msg["From"] = f"Infosys Portal <{EMAIL_ADDRESS}>"
    msg["To"] = to_addr
    msg.set_content(f"Your one-time code is {otp}. It expires in {OTP_TTL_SEC // 60} minutes. "
                    "If you did not request this, ignore this email.")
    msg.add_alternative(f"""<div style="font-family:Arial,sans-serif;max-width:420px;margin:auto;border:2px solid #272343;border-radius:14px;padding:24px">
      <h2 style="margin:0;color:#272343">⚡ Infosys Portal</h2>
      <p style="color:#475569">Use this code to reset your password:</p>
      <div style="font-size:34px;letter-spacing:10px;font-weight:700;background:#ffd803;color:#272343;text-align:center;border-radius:10px;padding:14px">{otp}</div>
      <p style="color:#64748b;font-size:13px">Expires in {OTP_TTL_SEC // 60} minutes. Never share this code with anyone.</p></div>""", subtype="html")
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=20) as s:
        s.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        s.send_message(msg)


def mask_email(e):
    n, d = e.split("@")
    return n[0] + "*" * max(len(n) - 2, 1) + n[-1] + "@" + d


# ───────────────────────── STATE & NAV ─────────────────────────
DEFAULTS = {"token": None, "page": "Login", "flash": None, "fails": 0, "lock_until": 0.0}
for k, v in DEFAULTS.items():
    st.session_state.setdefault(k, v)

FP_KEYS = ["sq_stage", "sq_user", "sq_question", "sq_tries", "otp_stage", "otp_email", "otp_hash",
           "otp_exp", "otp_tries", "otp_sent_at"]


def reset_fp():
    for k in FP_KEYS:
        st.session_state.pop(k, None)


def go(page, flash=None):
    st.query_params.clear()
    st.session_state.page = page
    st.session_state.flash = flash
    reset_fp()
    st.rerun()


def show_flash():
    if st.session_state.flash:
        kind, text = st.session_state.flash
        getattr(st, kind)(text)
        st.session_state.flash = None


def hero(heading, blurb):
    html(f"""<div class="hero">
      <div style="font-size:38px">⚡</div><h1>{heading}</h1><p>{blurb}</p>
      <ul><li>🔐 JWT-secured sessions</li><li>📧 Email OTP password recovery</li>
      <li>🛡️ bcrypt-hashed credentials</li><li>📊 Personalised user dashboard</li></ul></div>""")


def form_title(t, s):
    html(f'<p class="title">{t}</p><p class="sub">{s}</p>')


def two_pane(heading, blurb):
    left, right = st.columns([1, 1.1], gap="large")
    with left:
        hero(heading, blurb)
    return right


# ───────────────────────── PAGES: LOGGED OUT ─────────────────────────
def page_login():
    with two_pane("Infosys Portal", "Intelligent Analytics Portal. Sign in to access your dashboard, reports and insights."):
        form_title("Welcome back 👋", "Sign in to your account")
        show_flash()
        with st.form("f_login"):
            ident = st.text_input("Username or Email", placeholder="jane_doe or you@infosys.com").strip()
            pwd = st.text_input("Password", type="password", placeholder="Enter your password")
            ok = st.form_submit_button("Sign In →", type="primary")
        if ok:
            wait = st.session_state.lock_until - time.time()
            if wait > 0:
                st.error(f"⏳ Too many attempts. Try again in {int(wait) + 1}s.")
            elif not ident or not pwd:
                st.warning("⚠️ Username/Email and Password are both required.")
            else:
                row = q("SELECT username,email,password_hash FROM users WHERE email=? OR username=?",
                        (ident.lower(), ident), one=True)
                good = check_txt(pwd, row[2] if row else DUMMY_HASH) and row is not None
                if good:
                    st.session_state.token = make_jwt(row[0], row[1], "user")
                    st.session_state.fails = 0
                    st.rerun()
                st.session_state.fails += 1
                if st.session_state.fails >= 5:
                    st.session_state.lock_until, st.session_state.fails = time.time() + 30, 0
                st.error("❌ Invalid credentials.")   # generic on purpose
        c1, c2 = st.columns(2)
        if c1.button("Create account"): go("Signup")
        if c2.button("Forgot password?"): go("Forgot")


def page_signup():
    with two_pane("Join the portal", "Create your account in under a minute. Your security question lets you recover access later."):
        form_title("Create your account", "All fields are required")
        with st.form("f_signup"):
            a, b = st.columns(2)
            uname = a.text_input("Username", placeholder="jane_doe").strip()
            email = b.text_input("Email", placeholder="you@infosys.com").strip().lower()
            pw = st.text_input("Password", type="password", placeholder=PW_HINT)
            cpw = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security question", SECURITY_QUESTIONS)
            sa = st.text_input("Security answer", placeholder="Your answer").strip()
            ok = st.form_submit_button("Create account →", type="primary")
        if ok:
            err = None
            if not (uname and email and pw and cpw and sa):
                err = "All fields are mandatory."
            elif not valid_username(uname):
                err = "Username must be 3–30 characters: letters, numbers, . _ - only."
            elif not valid_email(email):
                err = "Enter a valid email like ab@cd.ef (2+ letters before @, between @ and dot, and after the dot)."
            else:
                err = check_new_password(pw, cpw)
            if err:
                st.error("❌ " + err)
            else:
                try:
                    q("INSERT INTO users(username,email,password_hash,security_question,security_answer_hash,created_at) VALUES(?,?,?,?,?,?)",
                      (uname, email, hash_txt(pw), sq, hash_txt(sa.lower()), dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")),
                      write=True)
                    go("Login", ("success", "✅ Account created! Please sign in."))
                except sqlite3.IntegrityError:
                    st.error("❌ That username or email is already registered.")
        if st.button("← Back to Sign In"): go("Login")


def reset_password(username_or_email_col, value, new_pw):
    q(f"UPDATE users SET password_hash=? WHERE {username_or_email_col}=?", (hash_txt(new_pw), value), write=True)


def page_forgot():
    with two_pane("Reset access", "Recover your account with your security question or a one-time code sent to your email."):
        form_title("Forgot password", "Choose how you want to verify it's you")
        show_flash()
        route = st.radio("Verification method", ["🔐 Security question", "📧 Email OTP"], horizontal=True,
                         key="fp_route", on_change=reset_fp, label_visibility="collapsed")
        if route.startswith("🔐"):
            forgot_sq()
        else:
            forgot_otp()
        if st.button("← Back to Sign In"): go("Login")


def forgot_sq():
    ss = st.session_state
    stage = ss.get("sq_stage", "user")
    if stage == "user":
        with st.form("f_sq_user"):
            u = st.text_input("Username", placeholder="Your username").strip()
            ok = st.form_submit_button("Continue →", type="primary")
        if ok:
            if not u:
                st.warning("⚠️ Username is required.")
            else:
                r = q("SELECT username,security_question FROM users WHERE username=?", (u,), one=True)
                if r:
                    ss.sq_user, ss.sq_question, ss.sq_tries, ss.sq_stage = r[0], r[1], 0, "answer"
                    st.rerun()
                st.error("❌ Username not found.")
    else:
        st.info(f"❓ **Security question:** {ss.sq_question}")
        with st.form("f_sq_answer"):
            ans = st.text_input("Your answer", placeholder="Security answer").strip().lower()
            npw = st.text_input("New password", type="password", placeholder=PW_HINT)
            cpw = st.text_input("Confirm new password", type="password")
            ok = st.form_submit_button("Reset password →", type="primary")
        if ok:
            err = None if ans else "Security answer is required."
            err = err or check_new_password(npw, cpw)
            if err:
                st.error("❌ " + err)
            else:
                r = q("SELECT security_answer_hash FROM users WHERE username=?", (ss.sq_user,), one=True)
                if r and check_txt(ans, r[0]):
                    reset_password("username", ss.sq_user, npw)
                    go("Login", ("success", "✅ Password updated. Please sign in with your new password."))
                ss.sq_tries += 1
                if ss.sq_tries >= 5:
                    reset_fp()
                    st.rerun()
                st.error(f"❌ Incorrect answer. {5 - ss.sq_tries} attempts left.")


def forgot_otp():
    ss = st.session_state
    stage = ss.get("otp_stage", "email")
    if stage == "email":
        if not (EMAIL_ADDRESS and EMAIL_PASSWORD):
            st.warning("📭 EMAIL_ADDRESS / EMAIL_PASSWORD secrets are not set, so OTP emails can't be sent.")
        with st.form("f_otp_email"):
            email = st.text_input("Registered email", placeholder="you@infosys.com").strip().lower()
            ok = st.form_submit_button("Send OTP →", type="primary")
        if ok:
            if not email:
                st.warning("⚠️ Email is required.")
            elif not valid_email(email):
                st.error("❌ Enter a valid email address.")
            elif not q("SELECT 1 FROM users WHERE email=?", (email,), one=True):
                st.error("❌ This email is not registered.")
            else:
                send_new_otp(email)
    elif stage == "verify":
        st.success(f"📨 A 6-digit code was sent to **{mask_email(ss.otp_email)}**. It expires in {OTP_TTL_SEC // 60} minutes.")
        with st.form("f_otp_verify"):
            code = st.text_input("Enter OTP", max_chars=6, placeholder="6-digit code").strip()
            ok = st.form_submit_button("Verify OTP →", type="primary")
        if ok:
            if not code:
                st.warning("⚠️ OTP is required.")
            elif time.time() > ss.otp_exp:
                st.error("⌛ OTP expired. Please request a new one.")
            elif check_txt(code, ss.otp_hash):
                ss.otp_stage = "reset"
                st.rerun()
            else:
                ss.otp_tries += 1
                if ss.otp_tries >= OTP_MAX_TRIES:
                    reset_fp()
                    go("Forgot", ("error", "🚫 Too many wrong attempts. Start again."))
                st.error(f"❌ Incorrect OTP. {OTP_MAX_TRIES - ss.otp_tries} attempts left.")
        c1, c2 = st.columns(2)
        cooldown = int(OTP_RESEND_SEC - (time.time() - ss.otp_sent_at))
        if c1.button("🔁 Resend OTP" if cooldown <= 0 else f"Resend in {cooldown}s", disabled=cooldown > 0):
            send_new_otp(ss.otp_email)
        if c2.button("Change email"):
            reset_fp()
            st.rerun()
    else:  # reset
        st.success("✅ OTP verified. Set your new password.")
        with st.form("f_otp_reset"):
            npw = st.text_input("New password", type="password", placeholder=PW_HINT)
            cpw = st.text_input("Confirm new password", type="password")
            ok = st.form_submit_button("Reset password →", type="primary")
        if ok:
            err = check_new_password(npw, cpw)
            if err:
                st.error("❌ " + err)
            else:
                reset_password("email", ss.otp_email, npw)
                go("Login", ("success", "✅ Password updated. Please sign in with your new password."))


def send_new_otp(email):
    ss = st.session_state
    otp = f"{secrets.randbelow(1_000_000):06d}"
    try:
        send_otp_email(email, otp)
    except smtplib.SMTPAuthenticationError:
        st.error("❌ Gmail rejected the login. Check EMAIL_ADDRESS and the 16-character App Password.")
        return
    except Exception as e:
        st.error(f"❌ Could not send email ({type(e).__name__}). Check your secrets and internet.")
        return
    ss.otp_email, ss.otp_hash = email, hash_txt(otp)
    ss.otp_exp, ss.otp_tries, ss.otp_sent_at = time.time() + OTP_TTL_SEC, 0, time.time()
    ss.otp_stage = "verify"
    st.rerun()


# ───────────────────────── PAGES: LOGGED IN ─────────────────────────
def sidebar(payload, items):
    with st.sidebar:
        role = "Member"
        html(f"""<div style="text-align:center;padding:14px 4px"><div style="font-size:30px">⚡</div>
          <div style="font-weight:700;font-family:Poppins;color:#272343">Infosys Portal</div>
          <div class="chip" style="margin-top:8px">{role}</div>
          <div style="font-size:12px;color:#64748b;margin-top:8px">{payload['sub']}</div></div>""")
        choice = st.radio("Navigate", items, key=f"nav_{payload['role']}")
        st.divider()
        if st.button("🚪 Logout"):
            st.session_state.token = None
            go("Login", ("success", "👋 You have been logged out."))
    return choice


def banner(sub, chip):
    html(f"""<div class="banner"><div><h1>⚡ Infosys Portal</h1><div class="s">{sub}</div></div>
      <div class="chip">{chip}</div></div>""")


def kpi(col, icon, val, label):
    with col:
        html(f'<div class="card kpi"><div class="i">{icon}</div><div class="v">{val}</div><div class="l">{label}</div></div>')


def page_user(p):
    page = sidebar(p, ["Overview", "Profile"])
    row = q("SELECT username,email,security_question,created_at FROM users WHERE email=?", (p["email"],), one=True)
    if not row:
        st.session_state.token = None
        go("Login", ("warning", "Account no longer exists."))
    banner("Analytics Dashboard", f"👤 {row[0]}")
    if page == "Overview":
        st.markdown(f"### 🎉 Welcome, {row[0]}!")
        st.caption(f"Signed in as {row[1]}")
        cols = st.columns(4)
        for c, (i, v, l) in zip(cols, [("📄", "128", "Documents Indexed"), ("🔍", "47", "Searches Today"),
                                       ("📊", "98.4%", "Efficiency Score"), ("🛡️", "Secured", "Session Status")]):
            kpi(c, i, v, l)
        st.write("")
        left, right = st.columns([1.2, 1])
        with left:
            fig = pgo.Figure(pgo.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index"},
                            gauge={"axis": {"range": [0, 100]}, "bar": {"color": "#ffd803"}, "bgcolor": "#bae8e8",
                                   "borderwidth": 2, "bordercolor": "#272343"}))
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", height=280, margin=dict(l=20, r=20, t=50, b=10),
                              font={"color": "#2d334a"})
            st.plotly_chart(fig)
        with right:
            exp = dt.datetime.fromtimestamp(p["exp"], dt.timezone.utc).strftime("%H:%M UTC")
            html(f"""<div class="card"><h4 style="margin-top:0">🔐 Session</h4>
              <p>Your JWT session is valid until <b>{exp}</b>.</p>
              <p style="color:#64748b;font-size:13px;margin:0">You'll be asked to sign in again after it expires.</p></div>""")
    else:
        st.markdown("### 👤 My Profile")
        html(f"""<div class="card"><p><b>Username:</b> {row[0]}</p><p><b>Email:</b> {row[1]}</p>
          <p><b>Security question:</b> {row[2]}</p><p style="margin:0"><b>Member since:</b> {row[3][:10]}</p></div>""")


# ───────────────────────── ROUTER ─────────────────────────
payload = verify_jwt(st.session_state.token) if st.session_state.token else None
if st.session_state.token and not payload:        # expired / tampered token
    st.session_state.token = None
    st.session_state.page, st.session_state.flash = "Login", ("warning", "⏰ Session expired. Please sign in again.")

if payload:
    page_user(payload)
else:
    st.markdown('<style>section[data-testid="stSidebar"],div[data-testid="stSidebarCollapsedControl"]{display:none!important}</style>',
                unsafe_allow_html=True)
    {"Login": page_login, "Signup": page_signup, "Forgot": page_forgot}.get(
        st.session_state.page, page_login)()

Writing app.py


### 4️⃣ Launch the app and get the public ngrok URL
Stop the cell (⏹) to shut everything down.

In [4]:
import os, time, subprocess
from pyngrok import ngrok, exception
from google.colab import userdata

def secret(*names):
    for n in names:
        try:
            v = userdata.get(n)
            if v:
                return v
        except Exception:
            pass
    raise SystemExit(f"❌ Missing Colab secret - add one of {names} (🔑) and enable notebook access.")

# Pass secrets to the Streamlit process through environment variables (never hard-coded)
os.environ["JWT_KEY"]     = secret("JWT_KEY", "JWT_SECRET")
os.environ["EMAIL_ADDRESS"]  = secret("EMAIL_ADDRESS")
os.environ["EMAIL_PASSWORD"] = secret("EMAIL_PASSWORD")

try:
    ngrok.set_auth_token(secret("ngrok_authentication_HF", "NGROK_AUTHTOKEN"))
except exception.PyngrokNgrokError as e:
    if "Authtoken already set" not in str(e):
        raise

# Light neo-brutalist theme
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\n'
            'secondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n\n[server]\nheadless=true\n')

# Clean up any previous run
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit run"])
time.sleep(1)

log = open("streamlit.log", "w")
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"],
                           stdout=log, stderr=log, env=os.environ.copy())
time.sleep(5)
if process.poll() is not None:
    print(open("streamlit.log").read())
    raise SystemExit("❌ Streamlit failed to start - see log above.")

public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal live at: {public_url}")
print("=" * 60)
print("⏳ Running... press the Stop button to shut down.")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    pass
finally:
    ngrok.kill()
    process.terminate()
    print("\n✅ ngrok tunnel closed and Streamlit stopped.")

🚀 Infosys Portal live at: https://storable-chef-july.ngrok-free.dev
⏳ Running... press the Stop button to shut down.

✅ ngrok tunnel closed and Streamlit stopped.
